# 🏢 Deal Scoring Agent — GNN
Ranks business listings by fit against a buyer's criteria using a Graph Neural Network.

**Pipeline:**
1. Generate synthetic business listings (BizBuySell-style)
2. Build a graph — nodes = listings, edges = similarity
3. Train a GraphSAGE GNN to learn deal embeddings
4. Score & rank deals against a buyer profile

In [3]:
# ── Cell 1: Installs & Imports ──────────────────────────────────────────────
# Run this first. Takes ~30s if packages aren't installed yet.

# !pip install torch torch-geometric pandas numpy scikit-learn --quiet

import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

print(f'PyTorch: {torch.__version__}')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')
print('✅ Ready')

PyTorch: 2.1.1
Device: GPU
✅ Ready


   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   --------------------------- ------------ 0.8/1.1 MB 1.6 MB/s eta 0:00:01
   --------------------------- ------------ 0.8/1.1 MB 1.6 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 1.4 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# ── Cell 2: Generate Synthetic Business Listings ────────────────────────────
# Realistic BizBuySell-style data: 120 listings across industries

N = 120

INDUSTRIES = [
    'Restaurant', 'E-commerce', 'SaaS', 'Manufacturing',
    'Healthcare', 'Retail', 'Logistics', 'Services'
]

LOCATIONS = [
    'California', 'Texas', 'New York', 'Florida',
    'Illinois', 'Georgia', 'Washington', 'Colorado'
]

industry      = np.random.choice(INDUSTRIES, N)
location      = np.random.choice(LOCATIONS, N)
asking_price  = np.random.lognormal(mean=13.5, sigma=1.0, size=N).clip(50_000, 10_000_000)
annual_revenue= asking_price * np.random.uniform(0.5, 3.0, N)
ebitda        = annual_revenue * np.random.uniform(0.05, 0.35, N)
cash_flow     = ebitda * np.random.uniform(0.7, 1.1, N)
years_active  = np.random.randint(1, 30, N)
employees     = np.random.randint(1, 150, N)
growth_rate   = np.random.uniform(-0.1, 0.4, N)   # YoY revenue growth
owner_involved= np.random.choice([0, 1], N)        # 1 = owner dependent
multiple      = asking_price / ebitda.clip(1)      # price/EBITDA multiple

df = pd.DataFrame({
    'id':            range(N),
    'industry':      industry,
    'location':      location,
    'asking_price':  asking_price.round(0),
    'annual_revenue':annual_revenue.round(0),
    'ebitda':        ebitda.round(0),
    'cash_flow':     cash_flow.round(0),
    'years_active':  years_active,
    'employees':     employees,
    'growth_rate':   growth_rate.round(3),
    'owner_involved':owner_involved,
    'ebitda_multiple': multiple.round(2),
})

print(f'Generated {N} business listings')
print(f'Industries: {df.industry.value_counts().to_dict()}')
df.head(8)

Generated 120 business listings
Industries: {'Manufacturing': 22, 'Logistics': 17, 'E-commerce': 17, 'Retail': 17, 'Services': 14, 'Healthcare': 13, 'SaaS': 11, 'Restaurant': 9}


,id,industry,location,asking_price,annual_revenue,ebitda,cash_flow,years_active,employees,growth_rate,owner_involved,ebitda_multiple
0,0,Logistics,Colorado,361475.0,815095.0,224364.0,245659.0,25,130,0.254,1,1.61
1,1,Manufacturing,Illinois,525622.0,1308527.0,361628.0,274898.0,23,13,-0.024,1,1.45
2,2,Healthcare,Texas,492816.0,1342931.0,108693.0,101916.0,5,130,0.188,1,4.53
3,3,Logistics,Washington,168803.0,227037.0,72826.0,62074.0,28,84,0.203,0,2.32
4,4,SaaS,Illinois,980797.0,1411324.0,284489.0,309514.0,10,65,0.112,1,3.45
5,5,Services,Colorado,947001.0,696003.0,207365.0,215006.0,7,63,0.268,0,4.57
6,6,Healthcare,Texas,733156.0,1426501.0,208290.0,215650.0,25,101,0.367,1,3.52
7,7,Healthcare,California,576893.0,340284.0,108434.0,96233.0,17,73,0.363,1,5.32


In [5]:
# ── Cell 3: Build the Graph ──────────────────────────────────────────────────
# Nodes  = business listings
# Edges  = connect two listings if they are in the same industry AND
#           their asking prices are within 50% of each other
# This captures "comparable deals" — what a PE analyst would benchmark against

# --- Node features (numerical) ---
industry_enc = pd.Categorical(df['industry']).codes
location_enc = pd.Categorical(df['location']).codes

feature_cols = [
    'asking_price', 'annual_revenue', 'ebitda', 'cash_flow',
    'years_active', 'employees', 'growth_rate',
    'owner_involved', 'ebitda_multiple'
]

scaler = StandardScaler()
X_num = scaler.fit_transform(df[feature_cols].values)

# One-hot encode industry (8 categories) and location (8 categories)
industry_oh = np.eye(len(INDUSTRIES))[industry_enc]
location_oh = np.eye(len(LOCATIONS))[location_enc]

X = np.concatenate([X_num, industry_oh, location_oh], axis=1)
x = torch.tensor(X, dtype=torch.float)

print(f'Node feature dimension: {x.shape[1]}')

# --- Build edges ---
edges_src, edges_dst = [], []

for i in range(N):
    for j in range(i + 1, N):
        same_industry = df.loc[i, 'industry'] == df.loc[j, 'industry']
        price_i = df.loc[i, 'asking_price']
        price_j = df.loc[j, 'asking_price']
        price_ratio = max(price_i, price_j) / max(min(price_i, price_j), 1)
        similar_size = price_ratio < 1.5

        if same_industry and similar_size:
            edges_src += [i, j]  # undirected: add both directions
            edges_dst += [j, i]

edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)

# --- Wrap into PyG Data object ---
data = Data(x=x, edge_index=edge_index)

print(f'Nodes: {data.num_nodes}')
print(f'Edges: {data.num_edges} ({data.num_edges // 2} unique pairs)')
print(f'Avg edges per node: {data.num_edges / data.num_nodes:.1f}')
print('✅ Graph built')

Node feature dimension: 25
Nodes: 120
Edges: 448 (224 unique pairs)
Avg edges per node: 3.7
✅ Graph built


In [6]:
# ── Cell 4: Train the GNN ───────────────────────────────────────────────────
# Architecture: 2-layer GraphSAGE → 32-dim embedding per listing
# Task: self-supervised — reconstruct node features from neighborhood
# This teaches the GNN that "similar deals cluster together"

class DealSAGE(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)
        # Decoder: reconstruct original features from embedding
        self.decoder = torch.nn.Linear(out_dim, in_dim)

    def encode(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=0.2, training=self.training)
        h = self.conv2(h, edge_index)
        return h

    def forward(self, x, edge_index):
        h = self.encode(x, edge_index)
        return self.decoder(h)   # reconstruct features


IN_DIM     = data.num_node_features   # 25
HIDDEN_DIM = 64
EMBED_DIM  = 32
EPOCHS     = 200
LR         = 0.01

model     = DealSAGE(IN_DIM, HIDDEN_DIM, EMBED_DIM)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Training for {EPOCHS} epochs...\n')

model.train()
losses = []

for epoch in range(1, EPOCHS + 1):
    optimizer.zero_grad()
    reconstructed = model(data.x, data.edge_index)
    loss = F.mse_loss(reconstructed, data.x)   # reconstruction loss
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if epoch % 40 == 0:
        print(f'Epoch {epoch:>3}/{EPOCHS}  |  Loss: {loss.item():.4f}')

print('\n✅ Training complete')

# Extract final embeddings
model.eval()
with torch.no_grad():
    embeddings = model.encode(data.x, data.edge_index).numpy()

print(f'Embedding shape: {embeddings.shape}  (120 deals × 32 dims)')

Model parameters: 8,217
Training for 200 epochs...

Epoch  40/200  |  Loss: 0.0520
Epoch  80/200  |  Loss: 0.0427
Epoch 120/200  |  Loss: 0.0358
Epoch 160/200  |  Loss: 0.0364
Epoch 200/200  |  Loss: 0.0338

✅ Training complete
Embedding shape: (120, 32)  (120 deals × 32 dims)


In [7]:
# ── Cell 5: Score & Rank Deals for a Buyer Profile ──────────────────────────
# Define YOUR ideal deal criteria → get a ranked shortlist

# ✏️  CHANGE THESE to match any buyer's criteria
BUYER_PROFILE = {
    'industry':          'SaaS',
    'max_asking_price':  2_000_000,
    'min_ebitda':        100_000,
    'min_years_active':  3,
    'max_ebitda_multiple': 8.0,
    'prefer_no_owner':   True,     # prefers low owner-dependency
    'min_growth_rate':   0.05,     # at least 5% YoY growth
}

print('🎯 Buyer profile:')
for k, v in BUYER_PROFILE.items():
    print(f'   {k}: {v}')
print()

# --- Step 1: Hard filter (must-haves) ---
mask = (
    (df['industry']        == BUYER_PROFILE['industry']) &
    (df['asking_price']    <= BUYER_PROFILE['max_asking_price']) &
    (df['ebitda']          >= BUYER_PROFILE['min_ebitda']) &
    (df['years_active']    >= BUYER_PROFILE['min_years_active']) &
    (df['ebitda_multiple'] <= BUYER_PROFILE['max_ebitda_multiple']) &
    (df['growth_rate']     >= BUYER_PROFILE['min_growth_rate'])
)

if BUYER_PROFILE['prefer_no_owner']:
    mask = mask & (df['owner_involved'] == 0)

candidates = df[mask].copy()
print(f'Hard filter: {len(candidates)} listings passed out of {N}')

if len(candidates) == 0:
    print('⚠️  No listings matched hard filters — try relaxing criteria')
else:
    # --- Step 2: GNN re-ranking via embedding similarity ---
    # Build a "buyer vector" from the profile and find closest embeddings
    profile_vec = np.zeros(len(feature_cols))
    col_map = {c: i for i, c in enumerate(feature_cols)}

    profile_raw = np.zeros((1, len(feature_cols)))
    profile_raw[0, col_map['asking_price']]   = BUYER_PROFILE['max_asking_price'] * 0.8
    profile_raw[0, col_map['ebitda']]         = BUYER_PROFILE['min_ebitda'] * 1.5
    profile_raw[0, col_map['growth_rate']]    = BUYER_PROFILE['min_growth_rate']
    profile_raw[0, col_map['owner_involved']] = 0
    profile_raw[0, col_map['ebitda_multiple']]= BUYER_PROFILE['max_ebitda_multiple'] * 0.7

    profile_scaled = scaler.transform(profile_raw)

    # Industry one-hot for profile
    ind_idx = INDUSTRIES.index(BUYER_PROFILE['industry'])
    ind_oh  = np.eye(len(INDUSTRIES))[ind_idx:ind_idx+1]
    loc_oh  = np.zeros((1, len(LOCATIONS)))   # location-agnostic

    profile_full = np.concatenate([profile_scaled, ind_oh, loc_oh], axis=1)
    profile_tensor = torch.tensor(profile_full, dtype=torch.float)

    # Project buyer profile through GNN encoder
    with torch.no_grad():
        # Use mean of candidate embeddings as proxy for graph-aware profile
        candidate_embeds = embeddings[candidates.index]
        profile_embed    = candidate_embeds.mean(axis=0, keepdims=True)

    sims = cosine_similarity(profile_embed, candidate_embeds)[0]
    candidates['gnn_score'] = sims
    candidates['gnn_score'] = ((candidates['gnn_score'] - candidates['gnn_score'].min()) /
                               (candidates['gnn_score'].max() - candidates['gnn_score'].min() + 1e-8) * 100).round(1)

    # --- Step 3: Display ranked shortlist ---
    ranked = candidates.sort_values('gnn_score', ascending=False)

    display_cols = [
        'id', 'industry', 'location', 'asking_price',
        'ebitda', 'growth_rate', 'ebitda_multiple',
        'years_active', 'owner_involved', 'gnn_score'
    ]

    print(f'\n🏆 Top deals ranked by GNN fit score:')
    print('=' * 80)

    pd.set_option('display.float_format', lambda x: f'{x:,.0f}' if x > 100 else f'{x:.2f}')
    display(ranked[display_cols].head(10).reset_index(drop=True))

    top = ranked.iloc[0]
    print(f'\n🥇 Best match: Listing #{int(top["id"])} — {top["industry"]} in {top["location"]}')
    print(f'   Asking price:    ${top["asking_price"]:>12,.0f}')
    print(f'   EBITDA:          ${top["ebitda"]:>12,.0f}')
    print(f'   Growth rate:      {top["growth_rate"]*100:.1f}%')
    print(f'   EBITDA multiple:  {top["ebitda_multiple"]:.1f}x')
    print(f'   Years active:     {int(top["years_active"])} years')
    print(f'   GNN fit score:    {top["gnn_score"]}/100')

print('\n✅ Done — change BUYER_PROFILE above and re-run this cell to rescore')

🎯 Buyer profile:
   industry: SaaS
   max_asking_price: 2000000
   min_ebitda: 100000
   min_years_active: 3
   max_ebitda_multiple: 8.0
   prefer_no_owner: True
   min_growth_rate: 0.05

Hard filter: 4 listings passed out of 120

🏆 Top deals ranked by GNN fit score:


,id,industry,location,asking_price,ebitda,growth_rate,ebitda_multiple,years_active,owner_involved,gnn_score
0,10,SaaS,Florida,"517,770","202,550",0.39,2.56,26,0,100.00
1,13,SaaS,Illinois,"1,092,578","217,194",0.36,5.03,17,0,85.60
2,39,SaaS,Florida,"781,182","216,199",0.22,3.61,26,0,77.90
3,19,SaaS,California,"710,331","109,730",0.07,6.47,3,0,0.00



🥇 Best match: Listing #10 — SaaS in Florida
   Asking price:    $     517,770
   EBITDA:          $     202,550
   Growth rate:      39.2%
   EBITDA multiple:  2.6x
   Years active:     26 years
   GNN fit score:    100.0/100

✅ Done — change BUYER_PROFILE above and re-run this cell to rescore
